In [1]:
# Project and symbolic tools
import heyoka as hy
import quivira as qv

# Numerics and visualization
import numpy as np
from _animation import animate_bar_model

In [2]:
model = qv.bar_joint_model()

# Number of cells in each direction
n_x = 7
n_y = 7

# Cell spacing
dx = 8
dy = 8

# Original cell geometry
base_point_pairs = [
    ((0, 0), (1, 4)),
    ((1, 4), (0, 8)),
    ((0, 8), (4, 7)),
    ((4, 7), (8, 8)),
    ((8, 8), (9, 4)),
    ((9, 4), (8, 0)),
    ((8, 0), (4, -1)),
    ((4, -1), (0, 0)),
]

# Keep track of already existing bars
existing_bars = set()

top_right_bar_index = None

for i in range(n_x):
    for j in range(n_y):

        offset_x = i * dx
        offset_y = j * dy

        for k, (start, end) in enumerate(base_point_pairs):

            new_start = (start[0] + offset_x, start[1] + offset_y)
            new_end = (end[0] + offset_x, end[1] + offset_y)

            bar_key = tuple(sorted((new_start, new_end)))

            if bar_key in existing_bars:
                continue

            # Identify bar 3 in the upper-right cell
            if i == n_x - 1 and j == n_y - 1 and k == 3:
                top_right_bar_index = len(model.bars)

            model.add_bar_from_point_pair(new_start, new_end)
            existing_bars.add(bar_key)

model.infer_joints()

model.add_support(0, -0.5, False, False)
model.add_external_force(top_right_bar_index, 0.5, 100, 100)
#model.add_external_force(3, 0, 0, 0, -30)

In [3]:
ode_system, W = model.build_lagrange_eom(return_W=True)

In [4]:
ta = hy.taylor_adaptive(ode_system, compact_mode=True)

In [6]:
state0 = model.get_initial_state()
model.check_constraint_fulfillment(state0)

In [7]:
ta.time = 0.0
ta.state[:] = state0

t_end = 1
tgrid = np.linspace(0.0, t_end, 401)
solution = ta.propagate_grid(tgrid)[-1]

In [8]:
# Use your existing 2D trajectory variable here:
animate_bar_model(model, solution, save_animation=True)